# 📑 End-to-End Smart Document Parser Pipeline (PDF → Markdown & JSON)

Notebook ini menyediakan **Unified Document Parsing Workflow** yang mengonversi dokumen PDF (baik Digital maupun Scan) menjadi **Markdown (.md)** dan **Structured JSON (.json)** secara otomatis.

---
### 📐 Arsitektur Pipeline (7-Layer Workflow):
1. **Layer 1**: Dependency Setup & Environment Initialization
2. **Layer 2**: Standard Data Models & Data Structures
3. **Layer 3**: Document Inspector & Router (Digital vs Scan Detection)
4. **Layer 4**: Multi-Engine Extraction Core (IBM Docling AI & Spatial Parser)
5. **Layer 5**: Unified Pipeline & Dual Exporter (Markdown + JSON Output)
6. **Layer 6**: Batch Document Processing Workflow
7. **Layer 7**: End-to-End Demo & Output Validation


---
# Layer 1: Dependency Setup & Environment Initialization

Memuat seluruh library ekosistem parser: , , , dan .


In [1]:
import os
import re
import json
from typing import List, Dict, Any, Optional
from dataclasses import dataclass, asdict
import pymupdf  # PyMuPDF
import pdfplumber
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, RapidOcrOptions

print("✅ Layer 1: Library & Environment Dependencies Berhasil Dimuat!")


✅ Layer 1: Library & Environment Dependencies Berhasil Dimuat!


---
# Layer 2: Core Data Models & Data Structures

Mendefinisikan struktur objek terpadu untuk merepresentasikan blok teks, tabel, halaman, dan dokumen utuh.


In [2]:
@dataclass
class TextBlock:
    text: str
    x0: float
    y0: float
    x1: float
    y1: float
    font_size: float = 10.0
    is_bold: bool = False
    page_num: int = 0
    block_type: str = "paragraph"
    
    @property
    def width(self) -> float:
        return self.x1 - self.x0
        
    @property
    def height(self) -> float:
        return self.y1 - self.y0
        
    @property
    def center_x(self) -> float:
        return (self.x0 + self.x1) / 2.0
        
    @property
    def center_y(self) -> float:
        return (self.y0 + self.y1) / 2.0

@dataclass
class TableBlock:
    page_num: int
    bbox: List[float]
    headers: List[str]
    rows: List[List[str]]
    markdown_table: str

@dataclass
class ParsedDocument:
    filename: str
    total_pages: int
    parser_engine: str
    markdown_content: str
    structured_json: Dict[str, Any]

print("✅ Layer 2: Data Models (TextBlock, TableBlock, ParsedDocument) Siap!")


✅ Layer 2: Data Models (TextBlock, TableBlock, ParsedDocument) Siap!


---
# Layer 3: Document Inspector & Smart Router

Menganalisis dokumen PDF untuk mendeteksi apakah dokumen berbasis Digital (Vector Text) atau Scan Gambar.


In [3]:
def inspect_pdf_type(pdf_path: str) -> Dict[str, Any]:
    """Menganalisis jenis dokumen PDF (Digital vs Scan)."""
    doc = pymupdf.open(pdf_path)
    total_pages = len(doc)
    total_text_len = 0
    has_images = False
    
    for page in doc:
        text = page.get_text()
        total_text_len += len(text.strip())
        if len(page.get_images()) > 0:
            has_images = True
            
    is_scanned = (total_text_len == 0)
    doc_type = "SCAN (Gambar)" if is_scanned else ("MIXED" if has_images else "DIGITAL (Teks)")
    
    return {
        "pdf_path": pdf_path,
        "total_pages": total_pages,
        "total_text_len": total_text_len,
        "is_scanned": is_scanned,
        "doc_type": doc_type
    }

print("✅ Layer 3: Document Inspector & Smart Router Siap!")


✅ Layer 3: Document Inspector & Smart Router Siap!


---
# Layer 4: Multi-Engine Extraction Core (IBM Docling AI & Spatial Parser)

Menyediakan dua engine ekstraksi tingkat tinggi:
1. **IBM Docling Engine**: AI Layout Transformer + RapidOCR (Rekomendasi Utama untuk Tabel & Form Kompleks).
2. **Custom Spatial Engine**: Coordinate-based Column & Typo Recovery Parser.


In [4]:
def extract_with_docling(pdf_path: str) -> Dict[str, Any]:
    """Ekstraksi menggunakan IBM Docling AI + RapidOCR Engine."""
    print(f"🚀 Executing IBM Docling AI Engine on: {pdf_path}")
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = True
    pipeline_options.ocr_options = RapidOcrOptions()
    
    converter = DocumentConverter(
        format_options={
            "pdf": PdfFormatOption(pipeline_options=pipeline_options)
        }
    )
    result = converter.convert(pdf_path)
    doc = result.document
    
    md_content = doc.export_to_markdown()
    raw_dict = doc.export_to_dict()
    
    return {
        "markdown": md_content,
        "raw_structure": raw_dict,
        "engine": "IBM Docling AI + RapidOCR"
    }

print("✅ Layer 4: Multi-Engine Extraction Core Siap!")


✅ Layer 4: Multi-Engine Extraction Core Siap!


---
# Layer 5: Unified Pipeline & Dual Exporter (Markdown + JSON Output)

Fungsi utama pipeline yang memproses dokumen PDF dari awal hingga memproduksi file **Markdown (.md)** dan **JSON (.json)** secara bersamaan.


In [5]:
def parse_pdf_to_markdown_and_json(
    pdf_path: str, 
    output_dir: str = ".", 
    save_files: bool = True
) -> ParsedDocument:
    """End-to-End Workflow: Konversi PDF ke Markdown & JSON secara bersamaan."""
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"File PDF tidak ditemukan: {pdf_path}")
        
    # Step 1: Inspeksi jenis PDF
    info = inspect_pdf_type(pdf_path)
    print(f"📌 Processing {os.path.basename(pdf_path)} [{info['doc_type']}] - {info['total_pages']} Halaman...")
    
    # Step 2: Ekstraksi dengan AI Engine
    extraction = extract_with_docling(pdf_path)
    
    # Step 3: Susun Objek JSON Terstruktur
    base_name = os.path.splitext(os.path.basename(pdf_path))[0]
    md_filename = f"{base_name}_parsed.md"
    json_filename = f"{base_name}_parsed.json"
    
    md_path = os.path.join(output_dir, md_filename)
    json_path = os.path.join(output_dir, json_filename)
    
    structured_json = {
        "document_info": info,
        "parser_engine": extraction["engine"],
        "total_pages": info["total_pages"],
        "markdown_content": extraction["markdown"],
        "document_hierarchy": extraction["raw_structure"]
    }
    
    # Step 4: Simpan File Output jika save_files=True
    if save_files:
        os.makedirs(output_dir, exist_ok=True)
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(extraction["markdown"])
            
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(structured_json, f, indent=2, ensure_ascii=False)
            
        print(f"  ✨ File Markdown tersimpan di : {md_path}")
        print(f"  ✨ File JSON tersimpan di     : {json_path}")
        
    return ParsedDocument(
        filename=os.path.basename(pdf_path),
        total_pages=info["total_pages"],
        parser_engine=extraction["engine"],
        markdown_content=extraction["markdown"],
        structured_json=structured_json
    )

print("✅ Layer 5: Unified Pipeline & Dual Exporter (Markdown + JSON) Siap!")


✅ Layer 5: Unified Pipeline & Dual Exporter (Markdown + JSON) Siap!


---
# Layer 6: Batch Document Processing Workflow

Memproses sekumpulan file PDF sekaligus dalam satu siklus kerja otomatis.


In [6]:
def batch_process_pdfs(pdf_files: List[str], output_dir: str = ".") -> List[ParsedDocument]:
    """Memproses banyak file PDF sekaligus dan menyimpan output Markdown & JSON."""
    results = []
    print(f"📦 Batch Processing {len(pdf_files)} Dokumen PDF...")
    for pdf in pdf_files:
        if os.path.exists(pdf):
            res = parse_pdf_to_markdown_and_json(pdf, output_dir=output_dir, save_files=True)
            results.append(res)
            print("-" * 60)
    return results

print("✅ Layer 6: Batch Processor Workflow Siap!")


✅ Layer 6: Batch Processor Workflow Siap!


---
# Layer 7: End-to-End Demo & Output Validation

Eksekusi penuh pipeline untuk 3 jenis dokumen:
1. **KL FULL SIGNED.pdf** (Dokumen Scan Kontrak 14 Halaman)
2. **nvidia10q.pdf** (Dokumen Laporan Keuangan Digital 48 Halaman)
3. **NP_diskominfo_aceh_tamiang_2026 (1).pdf** (Dokumen Scan Nota Pesanan + Tabel 3 Halaman)


In [7]:
# List dokumen target pengujian
TARGET_PDFS = [
    "NP_diskominfo_aceh_tamiang_2026 (1).pdf",
    "KL FULL SIGNED.pdf",
    "nvidia10q.pdf"
]

# Jalankan Workflow Pipeline Ekstraksi Lengkap (Menghasilkan .md dan .json)
parsed_docs = batch_process_pdfs(TARGET_PDFS, output_dir=".")

print("")
print("🎉 SELURUH PROSES EKSTRAKSI SELESAI!")
print(f"Total dokumen berhasil diproses: {len(parsed_docs)}")


📦 Batch Processing 3 Dokumen PDF...
📌 Processing NP_diskominfo_aceh_tamiang_2026 (1).pdf [SCAN (Gambar)] - 3 Halaman...
🚀 Executing IBM Docling AI Engine on: NP_diskominfo_aceh_tamiang_2026 (1).pdf
  ✨ File Markdown tersimpan di : ./NP_diskominfo_aceh_tamiang_2026 (1)_parsed.md
  ✨ File JSON tersimpan di     : ./NP_diskominfo_aceh_tamiang_2026 (1)_parsed.json
------------------------------------------------------------
📌 Processing KL FULL SIGNED.pdf [SCAN (Gambar)] - 14 Halaman...
🚀 Executing IBM Docling AI Engine on: KL FULL SIGNED.pdf
  ✨ File Markdown tersimpan di : ./KL FULL SIGNED_parsed.md
  ✨ File JSON tersimpan di     : ./KL FULL SIGNED_parsed.json
------------------------------------------------------------
📌 Processing nvidia10q.pdf [MIXED] - 48 Halaman...
🚀 Executing IBM Docling AI Engine on: nvidia10q.pdf
2026-09-03 15:53:46,444 MatchingPostProcessor WARNING  Orphan pdf_cell 108 recovered to col=1 by nearest-column fallback (row=1, x=703.5, dist=152.6)
2026-09-03 15:53:46,

[INFO] 2026-09-03 15:52:26,251 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-03 15:52:26,316 [RapidOCR] download_file.py:60: File exists and is valid: /Users/sutriadik24/Downloads/buildAParser/.venv311/lib/python3.11/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-09-03 15:52:26,316 [RapidOCR] main.py:63: Using /Users/sutriadik24/Downloads/buildAParser/.venv311/lib/python3.11/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-09-03 15:52:26,359 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-03 15:52:26,360 [RapidOCR] download_file.py:60: File exists and is valid: /Users/sutriadik24/Downloads/buildAParser/.venv311/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-03 15:52:26,360 [RapidOCR] main.py:63: Using /Users/sutriadik24/Downloads/buildAParser/.venv311/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-03 15:52:26